In [59]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
import re

# 데이터 재분류 함수
def reclassify_intent(question, original_intent):
    text = question.lower()
    
    # 퀴즈 카테고리를 기타로 재분류
    if original_intent == '퀴즈':
        return '기타'
    
    # 질문/문의 형태를 기타로 분류 (가장 중요한 수정사항)
    question_keywords = ['어디서', '언제', '어떻게', '무엇을', '누구', '왜', 
                        '신청은', '문의', '알려주', '가르쳐', '설명해',
                        '?', '？', '해요', '하나요', '인가요', '입니까', 
                        '됩니까', '하면', '가능한가', '방법']
    
    if any(keyword in text for keyword in question_keywords):
        return '기타'
    
    # 발표 관련 키워드 강화
    presentation_keywords = ['ppt', '프레젠테이션', '발표', '슬라이드', 
                           '발표자료', '발표문', '대본', 'presentation']
    if any(keyword in text for keyword in presentation_keywords):
        return '발표'
    
    # 요약 관련 키워드
    summary_keywords = ['요약', '핵심', '간단히', '짧게', '한눈에', '정리해', 
                       '간추려', '핵심만', '요점만']
    if any(keyword in text for keyword in summary_keywords):
        return '요약'
    
    # 보고서 관련 키워드
    report_keywords = ['보고서', '리포트', '목차', '문서로', '보고서로', 'report']
    if any(keyword in text for keyword in report_keywords):
        return '보고서'
    
    return original_intent

# 원본 데이터 로드 및 재분류
df = pd.read_csv("intent_dataset_100k.csv")
df['의도_수정'] = df.apply(lambda row: reclassify_intent(row['질문'], row['의도']), axis=1)

# 수정된 데이터만 사용
df_cleaned = df[['질문', '의도_수정']].copy()
df_cleaned.columns = ['질문', '의도']

# 수정된 데이터 저장
df_cleaned.to_csv("intent_dataset_cleaned.csv", index=False, encoding='utf-8')

print("수정된 데이터 분포:")
print(df_cleaned['의도'].value_counts())

수정된 데이터 분포:
의도
기타     51886
요약     23926
발표     16225
보고서     7983
Name: count, dtype: int64


In [60]:
# 텍스트 전처리 함수
def preprocess_text(text):
    # 특수문자 제거 (? 제외 - 중요한 특징)
    text = re.sub(r'[^\w\s?]', ' ', text)
    # 여러 공백을 하나로
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

# 데이터 전처리
X = df_cleaned['질문'].apply(preprocess_text)
y = df_cleaned['의도']

# 데이터 분할
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 개선된 TF-IDF 벡터라이저 설정
vectorizer = TfidfVectorizer(
    max_features=10000,     # 특성 수 증가
    ngram_range=(1, 2),     # 1-gram과 2-gram 사용
    min_df=2,               # 최소 문서 빈도
    max_df=0.95,            # 최대 문서 빈도
    stop_words=None         # 한국어는 따로 불용어 처리
)

X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# 여러 모델 비교
models = {
    'Naive Bayes': MultinomialNB(),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(kernel='linear', random_state=42)
}

best_model = None
best_score = 0

for name, model in models.items():
    model.fit(X_train_vec, y_train)
    score = model.score(X_test_vec, y_test)
    print(f"{name} 정확도: {score:.4f}")
    
    if score > best_score:
        best_score = score
        best_model = model

print(f"\n최고 성능 모델 정확도: {best_score:.4f}")

# 최고 성능 모델로 평가
y_pred = best_model.predict(X_test_vec)
print("\n분류 리포트:")
print(classification_report(y_test, y_pred))

# 혼동 행렬
print("\n혼동 행렬:")
print(confusion_matrix(y_test, y_pred))

Naive Bayes 정확도: 0.9999
Random Forest 정확도: 1.0000
SVM 정확도: 0.9999

최고 성능 모델 정확도: 1.0000

분류 리포트:
              precision    recall  f1-score   support

          기타       1.00      1.00      1.00     10377
          발표       1.00      1.00      1.00      3245
         보고서       1.00      1.00      1.00      1597
          요약       1.00      1.00      1.00      4785

    accuracy                           1.00     20004
   macro avg       1.00      1.00      1.00     20004
weighted avg       1.00      1.00      1.00     20004


혼동 행렬:
[[10377     0     0     0]
 [    0  3245     0     0]
 [    0     0  1597     0]
 [    1     0     0  4784]]


In [61]:
# 파이프라인 구성
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('classifier', MultinomialNB())
])

# 하이퍼파라미터 그리드
param_grid = {
    'tfidf__max_features': [5000, 10000, 15000],
    'tfidf__ngram_range': [(1, 1), (1, 2), (1, 3)],
    'tfidf__min_df': [1, 2, 3],
    'tfidf__max_df': [0.9, 0.95, 0.99],
    'classifier__alpha': [0.1, 0.5, 1.0, 1.5]
}

# 그리드 서치
grid_search = GridSearchCV(
    pipeline, 
    param_grid, 
    cv=5, 
    scoring='accuracy',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print(f"최적 파라미터: {grid_search.best_params_}")
print(f"최적 성능: {grid_search.best_score_:.4f}")

# 최적 모델로 최종 평가
best_pipeline = grid_search.best_estimator_
y_pred_best = best_pipeline.predict(X_test)

print("\n최적화된 모델 분류 리포트:")
print(classification_report(y_test, y_pred_best))

Fitting 5 folds for each of 324 candidates, totalling 1620 fits
최적 파라미터: {'classifier__alpha': 0.1, 'tfidf__max_df': 0.9, 'tfidf__max_features': 5000, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 1)}
최적 성능: 0.9999

최적화된 모델 분류 리포트:
              precision    recall  f1-score   support

          기타       1.00      1.00      1.00     10377
          발표       1.00      1.00      1.00      3245
         보고서       1.00      1.00      1.00      1597
          요약       1.00      1.00      1.00      4785

    accuracy                           1.00     20004
   macro avg       1.00      1.00      1.00     20004
weighted avg       1.00      1.00      1.00     20004



In [5]:
def predict_intent(text, model=best_pipeline):
    """텍스트의 의도를 예측하는 함수"""
    processed_text = preprocess_text(text)
    prediction = model.predict([processed_text])[0]
    probability = model.predict_proba([processed_text])[0]
    
    # 각 클래스별 확률
    classes = model.classes_
    prob_dict = dict(zip(classes, probability))
    
    return {
        'predicted_intent': prediction,
        'confidence': max(probability),
        'probabilities': prob_dict
    }

# 테스트 케이스
test_cases = [
    "휴가 신청은 어디서 해요?",
    "PPT 발표자료로 만들어줘",
    "핵심 내용을 요약해줘", 
    "보고서 형태로 작성해줘",
    "이메일로 작성해줘",
    "회의 내용을 간단히 정리해주세요",
    "프레젠테이션용 슬라이드 만들어주세요",
    "월간 보고서로 변환해주세요",
    "언제 회의가 있나요?",
    "어떻게 신청하면 되나요?"
]

print("예측 결과:")
print("="*60)
for text in test_cases:
    result = predict_intent(text)
    print(f"질문: {text}")
    print(f"예측: {result['predicted_intent']} (신뢰도: {result['confidence']:.3f})")
    print("-" * 40)

In [ ]:
# 클래스 불균형 해결을 위한 가중치 적용
from sklearn.utils.class_weight import compute_class_weight

# 클래스 가중치 계산
classes = np.unique(y_train)
class_weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight_dict = dict(zip(classes, class_weights))

# 가중치 적용 모델
weighted_model = RandomForestClassifier(
    n_estimators=100,
    class_weight=class_weight_dict,
    random_state=42
)

weighted_model.fit(X_train_vec, y_train)
y_pred_weighted = weighted_model.predict(X_test_vec)

print("가중치 적용 모델 성능:")
print(classification_report(y_test, y_pred_weighted))